# T7: second look (GPU) — more lines for right places?

For each photo, up to 5 candidates are looked at again (block `train/checks/second_look.py`): the pipeline's choice among the 6 standard queries,
its choice among all 27, and the 3 strongest other places (most inliers, places more than 4 sigma apart), all with >= 3 inliers. Only the dump is
read to pick them (no truth), so the notebook is safe to run on the blind set.

Variants per candidate: `recentre` (same drawing, tile centred on the corrected place), `aim_p` / `aim_m` (re-drawn with heading +- the fitted
angle and the scale divided by the fitted scale, tile re-centred). A smoke run decides which sign is right; later runs use `VARIANTS`.

Settings: `GATE_DATA`, `POOL_PHOTOS`, `DUMP_TAG` (folder in Test_Iteration_6/results/rich_dump), `VARIANTS` (comma list), `N_LIMIT`.

In [1]:
import os, sys, time, math
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
DATA = os.environ.get("GATE_DATA", "official")
POOL_PHOTOS = int(os.environ.get("POOL_PHOTOS", "500"))
DUMP_TAG = os.environ.get("DUMP_TAG", "official")
VARIANTS = os.environ.get("VARIANTS", "recentre,aim_p,aim_m").split(",")
N_LIMIT = int(os.environ.get("N_LIMIT", "0"))
DUMP = ROOT / "tests/Test_Iteration_6/results/rich_dump" / DUMP_TAG
OUT = ROOT / "tests/Test_Iteration_7/results/second_look"; OUT.mkdir(parents=True, exist_ok=True)
OUT_FILE = OUT / (os.environ.get("TAG", DUMP_TAG) + ".csv")
print(DATA, POOL_PHOTOS, DUMP_TAG, VARIANTS, N_LIMIT, OUT_FILE)

official 500 official ['recentre', 'aim_p', 'aim_m'] 12 /home/palash/GeoLoc/tests/Test_Iteration_7/results/second_look/smoke_official.csv


In [2]:
from train.checks.rich_dump import RichDumpPipeline, load_rich
pipeline = RichDumpPipeline(DATA, "edge_zncc", semantic_layer=True, semantic_decision=True, gamma=1.0, k_peaks=50, m_fine=10,
                            matcher="loftr_minima", min_inliers=8, ransac_pixels=None, matcher_confidence=0.22, satellite="new",
                            map_margin_m=None, loftr_coarse_threshold=0.21, dino_weight=None)
t0 = time.time(); pipeline.setup(POOL_PHOTOS, 7); print("setup %.0f s" % (time.time() - t0))
test_set = pipeline.test_set

level 3: 0.85 m/pixel, picture 16378 x 16005, map coverage 100%
level 4: 1.20 m/pixel, picture 11581 x 11317, map coverage 100%
level 5: 1.70 m/pixel, picture 8189 x 8002, map coverage 100%
level 6: 2.40 m/pixel, picture 5790 x 5658, map coverage 100%
level 7: 3.39 m/pixel, picture 4094 x 4001, map coverage 100%


map store ready, took 239 s


/home/palash/Ideaforge/GeoLoc/.venv-geoloc/lib/python3.10/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


/home/palash/Ideaforge/GeoLoc/.venv-geoloc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


setup 298 s


In [3]:
from train.checks.replay import STANDARD, WIDE, query_ids, pooled, choose
from train.checks.second_look import signed_fit, look_again

def pick_candidates(photo):
    """(query, rank, why) of up to 5 candidates with >= 3 inliers: std6 choice, wide27 choice, then the strongest other places."""
    picked = []
    def add(qi, r, why):
        if (qi, r) not in [(a, b) for a, b, _ in picked]:
            picked.append((qi, r, why))
    for why, keys in (("std6", STANDARD), ("wide27", WIDE)):
        ids = query_ids(photo, keys)
        cand = pooled(photo, ids)
        ok = cand["inliers"] >= 3
        if ok.any():
            sub = {k: v[ok] for k, v in cand.items()}
            k = choose(sub)
            add(int(sub["owner"][k]), int(sub["rank"][k]), why)
    cand = pooled(photo, query_ids(photo, WIDE))
    order = np.argsort(-cand["inliers"], kind="stable")
    places = [(photo["queries"][a]["peaks"][b, 1], photo["queries"][a]["peaks"][b, 2]) for a, b, _ in picked]
    for k in order:
        if len(picked) >= 5 or cand["inliers"][k] < 3:
            break
        x, y = cand["x"][k], cand["y"][k]
        if all(math.hypot(x - px, y - py) > 4 * photo["sigma"] for px, py in places):
            add(int(cand["owner"][k]), int(cand["rank"][k]), "strong"); places.append((x, y))
    return picked

files = sorted(DUMP.glob("photo_*.npz"), key=lambda p: int(p.stem.split("_")[1]))
if N_LIMIT:
    files = files[:N_LIMIT]
done = set(pd.read_csv(OUT_FILE).photo) if OUT_FILE.exists() else set()
print(len(files), "photos,", len(done), "done")

12 photos, 0 done


In [4]:
import torch
rows, t0, looks = [], time.time(), 0
for count, path in enumerate(files):
    pid = int(path.stem.split("_")[1])
    if pid in done:
        continue
    photo = load_rich(path)
    item = test_set.items[pid]
    assert abs(item["rel_height"] - photo["height"]) < 1e-3, "photo list differs from the dump"
    for qi, r, why in pick_candidates(photo):
        q = photo["queries"][qi]
        n, angle, fscale, (dx, dy) = signed_fit(q["matches"][r])
        x = q["peaks"][r, 1] + q["shifts"][r, 0]; y = q["peaks"][r, 2] + q["shifts"][r, 1]
        for variant in VARIANTS:
            sign = {"recentre": 0, "aim_p": 1, "aim_m": -1}[variant]
            power = 0 if variant == "recentre" else -1
            clock = time.time()
            try:
                s = look_again(pipeline, item, photo["level"], q["heading"], q["scale"], x, y, sign, angle, fscale, power)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); continue
            looks += 1
            rows.append(dict(photo=pid, query=qi, rank=r, why=why, variant=variant, first_inliers=float(q["inliers"][r]), first_angle=angle,
                             first_scale=fscale, x0=x, y0=y, seconds=time.time() - clock, **{"sl_" + k: v for k, v in s.items()}))
    if count % 25 == 0 and rows:
        pd.DataFrame(rows).to_csv(OUT_FILE, mode="a", header=not OUT_FILE.exists(), index=False); rows = []
        print("photo %d/%d | %.0f s | %.3f s per look" % (count, len(files), time.time() - t0, (time.time() - t0) / max(looks, 1)), flush=True)
if rows:
    pd.DataFrame(rows).to_csv(OUT_FILE, mode="a", header=not OUT_FILE.exists(), index=False)
print("done %.0f s, %d looks" % (time.time() - t0, looks))

photo 0/12 | 1 s | 0.148 s per look


done 6 s, 81 looks


In [5]:
res = pd.read_csv(OUT_FILE)
print(res.groupby("variant")[["first_inliers", "sl_pl_inliers", "seconds"]].mean().round(3))

          first_inliers  sl_pl_inliers  seconds
variant                                        
aim_m            26.111         17.926    0.042
aim_p            26.111         23.852    0.044
recentre         26.111         25.481    0.066
